# CoCa C1@585 vs C4@585 formal classifier runs


In [ ]:
from pathlib import Path
ACCOUNT_LABEL = ""  # A, B, or C
RUN_MODE = "fresh"  # fresh or resume; account switches use resume
EXPECTED_GIT_COMMIT = "b584bd321dd11258469f8c564bcc8a82a3ae11ac"
RUN_VERSION = "v1_safe_v2"
CHECKPOINT_FORMAT = "frozen_backbone_head_only_v1"
MAX_COCA_CHECKPOINT_BYTES = 100 * 1024 * 1024
STALE_MARKER_CONFIRMATION = ""  # Only after the previous runtime is confirmed stopped: CLEAR STALE MARKER
REPO_URL = "https://github.com/sfczaa/ddpm-derm-augmentation.git"
SHARED_PROJECT_DIR = Path("/content/drive/MyDrive/ddpm-derm-augmentation")
SHARED_RUN_ROOT = Path("/content/drive/MyDrive/ddpm-derm-coca-runs")
COCA_ROOT = SHARED_RUN_ROOT / "sqrt_balanced_seed0_v1" / "coca_classifier" / RUN_VERSION
VALIDATION_RECORD = COCA_ROOT / "validation_record.json"
FORMAL_ROOT = COCA_ROOT / "formal"
CHECKPOINT_ROOT = FORMAL_ROOT / "checkpoints" / "coca_vit_b32"
RESULTS_ROOT = FORMAL_ROOT / "results" / "coca_vit_b32"
RECORDS_ROOT = FORMAL_ROOT / "records"
EXECUTED_NOTEBOOKS_ROOT = FORMAL_ROOT / "executed_notebooks"
RUNNING_MARKER = FORMAL_ROOT / "_RUNNING.json"
COMPLETED_MARKER = FORMAL_ROOT / "_COMPLETED.json"
FORMAL_IDENTITY_RECORD = RECORDS_ROOT / "formal_identity.json"
HANDOFF_HISTORY_RECORD = RECORDS_ROOT / "handoff_history.json"
SHARED_ROOT_SENTINEL = SHARED_RUN_ROOT / ".coca_shared_root.json"
CANDIDATE_MANIFEST = SHARED_PROJECT_DIR / "outputs" / "exploratory_balanced_ddpm" / "sqrt_balanced_seed0_v1" / "candidate_synthetic_df" / "epoch0100_seed0" / "synthetic_df.csv"
EXPECTED_CANDIDATE_SHA256 = "9ef9b44e404f74aab8211f4e7d123da3258ba8ba4e3004a4147d1761ed343b34"
assert ACCOUNT_LABEL in {"A", "B", "C"}
assert RUN_MODE in {"fresh", "resume"}
assert len(EXPECTED_GIT_COMMIT) == 40 and EXPECTED_GIT_COMMIT != "REPLACE_AFTER_PUSH"

## Phase 0: Code, validation, data, and dependency checks


In [ ]:
import hashlib, json, os, shutil, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
assert SHARED_PROJECT_DIR.is_dir(), f"missing shared project shortcut: {SHARED_PROJECT_DIR}"
assert SHARED_RUN_ROOT.is_dir(), f"missing shared run shortcut; do not create a private replacement: {SHARED_RUN_ROOT}"
subprocess.run(["nvidia-smi"], check=True)
CODE_DIR = Path("/content/ddpm-coca-code")
assert not CODE_DIR.exists(), f"fresh runtime setup required: {CODE_DIR}"
subprocess.run(["git", "clone", REPO_URL, str(CODE_DIR)], check=True)
subprocess.run(["git", "-C", str(CODE_DIR), "checkout", "--detach", EXPECTED_GIT_COMMIT], check=True)
commit = subprocess.check_output(["git", "-C", str(CODE_DIR), "rev-parse", "HEAD"], text=True).strip()
status = subprocess.check_output(["git", "-C", str(CODE_DIR), "status", "--short"], text=True).strip()
assert commit == EXPECTED_GIT_COMMIT and not status
os.environ["HF_HOME"] = "/content/hf-cache"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch==3.3.0", "pandas>=2.0", "pillow>=12.3.0"], check=True)
sys.path.insert(0, str(CODE_DIR / "src"))
from ddpm_derm.notebook_runtime import require_training_runtime
require_training_runtime()
from importlib.metadata import version
import pandas as pd
import torch
from ddpm_derm import coca_run
assert torch.cuda.is_available() and version("open_clip_torch") == "3.3.0"
resolved_root = coca_run.require_existing_shared_root(SHARED_RUN_ROOT)
drive_probe = coca_run.probe_shared_drive(resolved_root)
assert SHARED_ROOT_SENTINEL.is_file() and VALIDATION_RECORD.is_file()
sentinel = json.loads(SHARED_ROOT_SENTINEL.read_text(encoding="utf-8"))
validation = json.loads(VALIDATION_RECORD.read_text(encoding="utf-8"))
assert sentinel["shared_root_uuid"] == validation["shared_root_uuid"]
assert sentinel["resolved_path"] == str(resolved_root) and sentinel["run_version"] == "v1"
if sentinel.get("drive_folder_id") is not None: assert sentinel["drive_folder_id"] == validation["shared_root_identity"]["drive_folder_id"]
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""): digest.update(chunk)
    return digest.hexdigest()
assert CANDIDATE_MANIFEST.is_file() and sha256(CANDIDATE_MANIFEST) == EXPECTED_CANDIDATE_SHA256
LOCAL_DATA_DIR = Path("/content/ham10000-data")
assert not LOCAL_DATA_DIR.exists(); shutil.copytree(SHARED_PROJECT_DIR / "data", LOCAL_DATA_DIR)
os.environ["DDPM_DERM_DATA_DIR"] = str(LOCAL_DATA_DIR)
frames = {split: pd.read_csv(LOCAL_DATA_DIR / "manifests" / f"{split}.csv") for split in ("train", "val", "test")}
assert {key: len(value) for key, value in frames.items()} == {"train": 6995, "val": 1510, "test": 1510}
assert int((frames["train"]["dx"] == "df").sum()) == 85
fixed_split_identity = sha256(LOCAL_DATA_DIR / "manifests" / "train.csv")
formal_output_identity = f"{sentinel['shared_root_uuid']}:sqrt_balanced_seed0_v1:coca_classifier:{RUN_VERSION}:formal"
expected_validation = {"git_commit": commit, "run_version": RUN_VERSION, "candidate_manifest_sha256": EXPECTED_CANDIDATE_SHA256, "fixed_split_identity": fixed_split_identity, "shared_root_uuid": sentinel["shared_root_uuid"], "formal_output_identity": formal_output_identity, "checkpoint_format": CHECKPOINT_FORMAT, "encoder_weights_stored": False}
coca_run.require_validation_record(validation, expected_validation)
print({"validation": validation["validation_status"], "formal_training_started": validation["formal_training_started"], "gpu": torch.cuda.get_device_name(0), "drive_probe": drive_probe})

In [ ]:
from PIL import Image
from ddpm_derm.model import build_model, model_identity
sanity_model = build_model(arch="coca_vit_b32", freeze_backbone=True, coca_pretrained="laion2b_s13b_b90k").cuda()
sample_path = LOCAL_DATA_DIR / frames["train"].iloc[0]["image_path"]
sample = sanity_model.eval_preprocess(Image.open(sample_path).convert("RGB")).unsqueeze(0).cuda()
sanity_model.train(); sanity_logits = sanity_model(sample)
assert tuple(sanity_logits.shape) == (1, 7) and not sanity_model.encoder.training
current_model_identity = model_identity(sanity_model, "coca_vit_b32", 128)
assert current_model_identity == validation["model_identity"]
del sanity_model, sanity_logits, sample; torch.cuda.empty_cache()
print("single-batch sanity and validation model identity: OK")

## Phase 1: Fresh/resume scan and run marker


In [ ]:
from datetime import datetime, timezone
if RUNNING_MARKER.exists():
    print(RUNNING_MARKER.read_text(encoding="utf-8"))
    if STALE_MARKER_CONFIRMATION != "CLEAR STALE MARKER": raise RuntimeError("existing marker retained; confirm the old runtime is stopped")
    coca_run.clear_stale_marker(RUNNING_MARKER, STALE_MARKER_CONFIRMATION)
if RUN_MODE == "fresh":
    existing = [] if not FORMAL_ROOT.exists() else [p for p in FORMAL_ROOT.rglob("*") if p.is_file() and (p.suffix in {".pt", ".json"} or p.name.startswith("_"))]
    assert not existing, f"fresh mode refuses existing formal artifacts: {existing[:10]}"
else:
    assert FORMAL_ROOT.is_dir(), f"resume mode requires existing formal root: {FORMAL_ROOT}"
for path in (FORMAL_ROOT, CHECKPOINT_ROOT, RESULTS_ROOT, RECORDS_ROOT, EXECUTED_NOTEBOOKS_ROOT): coca_run.ensure_tree(SHARED_RUN_ROOT, path.relative_to(SHARED_RUN_ROOT))
formal_identity = {**expected_validation, "model_identity": current_model_identity}
if RUN_MODE == "fresh": coca_run.write_json_atomic(FORMAL_IDENTITY_RECORD, formal_identity)
else:
    assert FORMAL_IDENTITY_RECORD.is_file(), f"resume identity record missing: {FORMAL_IDENTITY_RECORD}"
    coca_run.require_resume_identity(json.loads(FORMAL_IDENTITY_RECORD.read_text(encoding="utf-8")), formal_identity)
marker = coca_run.session_marker(ACCOUNT_LABEL, RUN_MODE, {**formal_identity, "current_variant": None, "current_seed": None, "current_epoch": 0, "resolved_shared_root": str(resolved_root), "model_name": "coca_ViT-B-32", "pretrained_tag": "laion2b_s13b_b90k"})
coca_run.create_running_marker(RUNNING_MARKER, marker)
handoff_history = json.loads(HANDOFF_HISTORY_RECORD.read_text(encoding="utf-8"))["sessions"] if HANDOFF_HISTORY_RECORD.is_file() else []
handoff_history.append({"account_label": ACCOUNT_LABEL, "session_id": marker["session_id"], "hostname": marker["hostname"], "run_mode": RUN_MODE, "started_utc": marker["started_utc"]})
coca_run.write_json_atomic(HANDOFF_HISTORY_RECORD, {"sessions": handoff_history})
run_queue = [(variant, seed) for variant in ("C1", "C4") for seed in (0, 1, 2)]
print("fixed queue:", run_queue)
print("checkpoint cadence: every completed epoch; worst-case loss: one unfinished epoch")

## Phase 2: Six 20-epoch runs with resume


In [ ]:
from ddpm_derm.checkpoint import load_checkpoint

import re, time
env = os.environ.copy(); env["PYTHONPATH"] = str(CODE_DIR / "src"); env["PYTHONUNBUFFERED"] = "1"
def paths_for(variant, seed):
    root = CHECKPOINT_ROOT / f"{variant}_seed{seed}"
    return root / "best.pt", root / "last.pt", RESULTS_ROOT / f"results_{variant}_seed{seed}.json"
def validate_completed(variant, seed):
    best, last, result_path = paths_for(variant, seed)
    assert best.is_file() and last.is_file() and result_path.is_file()
    result = json.loads(result_path.read_text(encoding="utf-8"))
    assert result["variant"] == variant and result["seed"] == seed and len(result["history"]) == 20
    assert result["data_counts"] == {"train": 7495, "val": 1510, "test": 1510}
    assert result["run_identity"]["model_identity"] == current_model_identity
    assert result["run_identity"]["candidate_manifest_sha256"] == EXPECTED_CANDIDATE_SHA256
    assert result["checkpoint_format"] == CHECKPOINT_FORMAT and result["run_identity"]["checkpoint_format"] == CHECKPOINT_FORMAT
    assert result["encoder_weights_stored"] is False
    for checkpoint_path in (best, last):
        checkpoint = load_checkpoint(checkpoint_path, map_location="cpu")
        assert checkpoint["run_identity"] == result["run_identity"]
        assert checkpoint["checkpoint_format"] == CHECKPOINT_FORMAT
        assert "head_state_dict" in checkpoint and "model_state_dict" not in checkpoint and "encoder_state_dict" not in checkpoint
        assert set(checkpoint["head_state_dict"]) == {"weight", "bias"}
        assert not any(any(token in key.lower() for token in ("encoder", "text", "caption", "decoder", "visual")) for key in checkpoint)
        coca_run.checkpoint_size(checkpoint_path, arch="coca_vit_b32")
        assert checkpoint_path.stat().st_size <= MAX_COCA_CHECKPOINT_BYTES
        assert len(checkpoint["history"]) == checkpoint["epoch"] and 1 <= checkpoint["epoch"] <= 20
    assert result["checkpoint_sizes"] == {"best_pt_bytes": best.stat().st_size, "last_pt_bytes": last.stat().st_size}
    assert load_checkpoint(last, map_location="cpu")["epoch"] == 20
    return result
def update_marker(**updates):
    state = json.loads(RUNNING_MARKER.read_text(encoding="utf-8")); state.update(updates, last_updated_utc=coca_run.utc_now()); coca_run.write_json_atomic(RUNNING_MARKER, state)
for variant, seed in run_queue:
    best, last, result_path = paths_for(variant, seed)
    if result_path.exists():
        validate_completed(variant, seed); print(f"[{variant} seed {seed}] completed identity verified; skip"); continue
    update_marker(current_variant=variant, current_seed=seed, current_epoch=0)
    command = [sys.executable, "-u", "-m", "ddpm_derm.train_classifier", "--arch", "coca_vit_b32", "--freeze-backbone", "--coca-pretrained", "laion2b_s13b_b90k", "--variant", variant, "--seed", str(seed), "--epochs", "20", "--batch-size", "32", "--lr", "3e-4", "--weight-decay", "1e-4", "--df-target-count", "585", "--num-workers", "2", "--output-dir", str(FORMAL_ROOT), "--run-label", "coca_formal_v1", "--run-version", RUN_VERSION, "--shared-root-uuid", sentinel["shared_root_uuid"], "--formal-output-identity", formal_output_identity, "--fixed-split-identity", fixed_split_identity, "--candidate-sha256", EXPECTED_CANDIDATE_SHA256]
    if variant == "C4": command += ["--generated-manifest", str(CANDIDATE_MANIFEST)]
    if RUN_MODE == "resume": command.append("--resume")
    process = subprocess.Popen(command, cwd=CODE_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
        match = re.search(r"\[epoch (\d+)/20\]", line)
        if match: update_marker(current_epoch=int(match.group(1)))
    if process.wait(): raise RuntimeError(f"formal run failed: {variant} seed {seed}")
    validate_completed(variant, seed)
    print(f"[{variant} seed {seed}] result, best.pt, last.pt reopened and verified")

## Phases 3-4: Post-run check and aggregation


In [ ]:
runs = [validate_completed(variant, seed) for variant, seed in run_queue]
assert len(list(RESULTS_ROOT.glob("results_*.json"))) == 6
aggregate = coca_run.aggregate_results(runs)
assert aggregate["ddof"] == 0
aggregate_path = RECORDS_ROOT / "aggregate_results.json"
coca_run.write_json_atomic(aggregate_path, aggregate)
print(json.dumps(aggregate, indent=2))
print("Interpret only within-CoCa C4-C1 direction; do not rank absolute CoCa versus ResNet scores.")

## Phase 5: Completion records and archive


In [ ]:
checkpoint_paths = {f"{variant}_seed{seed}": {"best": str(paths_for(variant, seed)[0]), "last": str(paths_for(variant, seed)[1]), "result": str(paths_for(variant, seed)[2]), "best_pt_bytes": paths_for(variant, seed)[0].stat().st_size, "last_pt_bytes": paths_for(variant, seed)[1].stat().st_size} for variant, seed in run_queue}
training_record = {"status": "completed", "runs": [{"variant": variant, "seed": seed} for variant, seed in run_queue], "artifact_paths": checkpoint_paths, "shared_root_identity": sentinel, "git_commit": commit, "dependency_versions": {"open_clip_torch": version("open_clip_torch"), "torch": torch.__version__}, "model_identity": current_model_identity, "checkpoint_format": CHECKPOINT_FORMAT, "checkpoint_sizes": {key: {"best_pt_bytes": value["best_pt_bytes"], "last_pt_bytes": value["last_pt_bytes"]} for key, value in checkpoint_paths.items()}, "encoder_weights_stored": False, "candidate_manifest_sha256": EXPECTED_CANDIDATE_SHA256, "aggregate_results": aggregate, "validation_record": str(VALIDATION_RECORD), "start_utc": json.loads(RUNNING_MARKER.read_text(encoding="utf-8"))["started_utc"], "end_utc": coca_run.utc_now(), "account_session_handoff_history": json.loads(HANDOFF_HISTORY_RECORD.read_text(encoding="utf-8"))["sessions"], "executed_notebook_archive_path": str(EXECUTED_NOTEBOOKS_ROOT / "colab_coca_classifier_executed.ipynb")}
training_record_path = RECORDS_ROOT / "training_record.json"
coca_run.write_json_atomic(training_record_path, training_record)
coca_run.write_json_atomic(COMPLETED_MARKER, training_record)
RUNNING_MARKER.unlink()
assert COMPLETED_MARKER.is_file() and not RUNNING_MARKER.exists()
print("FORMAL COCA RUNS COMPLETED AND VERIFIED")
print("Archive the executed notebook at:", training_record["executed_notebook_archive_path"])
print("Do not add the executed notebook, checkpoints, results, or model weights to Git.")